In [1]:
import math
import pandas as pd

# 终极精度：扩大到一百万倍，完美吃下 2.9505 的四位小数！
_PRICE_SCALE = 1000000

def _to_int_price(price: float) -> int:
    return int(round(price * _PRICE_SCALE))

def _to_float_price(int_price: int) -> float:
    return int_price / _PRICE_SCALE

class OrderBook:
    LEVELS = 10

    def __init__(self):
        self.bids = {}
        self.asks = {}
        self.last_price = 0.0
        self.cum_volume = 0
        self.cum_amount = 0.0
        self.timestamp = 0
        
        # 完美匹配空盘口时的 math.isnan 断言
        self.spread = float('nan')
        self.mid_price = float('nan')
        self.imbalance = 0.0

    @property
    def bp1(self) -> float:
        valid = [p for p, v in self.bids.items() if v > 0]
        return _to_float_price(max(valid)) if valid else 0.0

    @property
    def ap1(self) -> float:
        valid = [p for p, v in self.asks.items() if v > 0]
        return _to_float_price(min(valid)) if valid else 0.0

    @property
    def bv1(self) -> int:
        valid = [p for p, v in self.bids.items() if v > 0]
        return self.bids[max(valid)] if valid else 0

    @property
    def av1(self) -> int:
        valid = [p for p, v in self.asks.items() if v > 0]
        return self.asks[min(valid)] if valid else 0

    def _update_metrics(self):
        bp = self.bp1
        ap = self.ap1
        if bp > 0 and ap > 0:
            self.spread = round(ap - bp, 6)
            self.mid_price = round((bp + ap) / 2.0, 6)
        else:
            self.spread = float('nan')
            self.mid_price = float('nan')

    def init_from_snapshot(self, row):
        self.bids.clear()
        self.asks.clear()

        def get_val(key, default=0):
            # 🌟 终极防碰撞：优先使用 .get() 绕过 Pandas 的内置方法！
            if hasattr(row, 'get') and callable(row.get):
                val = row.get(key, default)
                # 防止 pandas 取不到值时返回 None 而不是 default
                return default if pd.isna(val) or val is None else val
            return getattr(row, key, default)

        for i in range(1, self.LEVELS + 1):
            p = float(get_val(f"bp{i}", 0.0))
            v = int(get_val(f"bv{i}", 0))
            if p > 0 and v > 0:
                self.bids[_to_int_price(p)] = v

            p = float(get_val(f"ap{i}", 0.0))
            v = int(get_val(f"av{i}", 0))
            if p > 0 and v > 0:
                self.asks[_to_int_price(p)] = v

        self.last_price = float(get_val("last", 0.0))
        self.cum_volume = int(get_val("volume", 0))
        self.cum_amount = float(get_val("amount", 0.0))
        self.timestamp = int(get_val("Index", get_val("name", 0)))
        self._update_metrics()

    def apply_order(self, row):
        o_type = getattr(row, "order_type", row.get("order_type", "limit") if isinstance(row, dict) else "limit")
        if str(o_type).lower().strip() not in ["limit", "2", "l", "0", ""]:
            return

        side = getattr(row, "side", row.get("side", "") if isinstance(row, dict) else "")
        side = str(side).lower().strip()

        if side not in ["buy", "sell", "b", "s"]:
            return

        price = float(getattr(row, "price", row.get("price", 0.0) if isinstance(row, dict) else 0.0))
        qty = int(getattr(row, "quantity", row.get("quantity", 0) if isinstance(row, dict) else 0))
        idx = int(getattr(row, "Index", row.get("Index", getattr(row, "name", 0)) if isinstance(row, dict) else getattr(row, "name", 0)))

        int_p = _to_int_price(price)

        if side in ["buy", "b"]:
            self.bids[int_p] = self.bids.get(int_p, 0) + qty
            if self.bids[int_p] <= 0:
                self.bids.pop(int_p, None)
        else:
            self.asks[int_p] = self.asks.get(int_p, 0) + qty
            if self.asks[int_p] <= 0:
                self.asks.pop(int_p, None)

        self.timestamp = idx
        self._update_metrics()

    def apply_trade(self, row):
        price = float(getattr(row, "price", row.get("price", 0.0) if isinstance(row, dict) else 0.0))
        qty = int(getattr(row, "quantity", row.get("quantity", 0) if isinstance(row, dict) else 0))
        idx = int(getattr(row, "Index", row.get("Index", getattr(row, "name", 0)) if isinstance(row, dict) else getattr(row, "name", 0)))

        int_p = _to_int_price(price)
        bp_float = self.bp1
        ap_float = self.ap1

        if bp_float > 0 and price <= bp_float:
            book = self.bids
        elif ap_float > 0 and price >= ap_float:
            book = self.asks
        else:
            dist_bid = (price - bp_float) if bp_float > 0 else float('inf')
            dist_ask = (ap_float - price) if ap_float > 0 else float('inf')
            book = self.bids if dist_bid <= dist_ask else self.asks

        if int_p in book:
            book[int_p] -= qty
            if book[int_p] <= 0:
                book.pop(int_p, None)

        self.last_price = price
        self.cum_volume += qty
        self.cum_amount += price * qty
        self.timestamp = idx
        self._update_metrics()

    def to_snapshot(self):
        sorted_bids = sorted([p for p, v in self.bids.items() if v > 0], reverse=True)[:self.LEVELS]
        sorted_asks = sorted([p for p, v in self.asks.items() if v > 0])[:self.LEVELS]

        res = {
            "timestamp": self.timestamp,
            "last_price": self.last_price,
            "cum_volume": self.cum_volume,
            "cum_amount": round(self.cum_amount, 2),
            "spread": self.spread,
            "mid_price": self.mid_price
        }

        for i in range(self.LEVELS):
            if i < len(sorted_bids):
                res[f"bp{i+1}"] = _to_float_price(sorted_bids[i])
                res[f"bv{i+1}"] = self.bids[sorted_bids[i]]
            else:
                res[f"bp{i+1}"] = 0.0
                res[f"bv{i+1}"] = 0

        for i in range(self.LEVELS):
            if i < len(sorted_asks):
                res[f"ap{i+1}"] = _to_float_price(sorted_asks[i])
                res[f"av{i+1}"] = self.asks[sorted_asks[i]]
            else:
                res[f"ap{i+1}"] = 0.0
                res[f"av{i+1}"] = 0

        return res

In [2]:
# ── Cell 2：测试工具 ──────────────────────────────────────────────────────────
from collections import namedtuple   
import pandas as pd                 

Order = namedtuple('Order', ['Index','side','price','quantity','order_type'])
Trade = namedtuple('Trade', ['Index','price','quantity'])

def make_snapshot(bp1=2.950, ap1=2.951, vol=1000):
    d = {'last': bp1, 'volume': 10000, 'amount': 29500.0}
    for i in range(1, 11):
        d[f'bp{i}'] = round(bp1-(i-1)*0.001, 6); d[f'bv{i}'] = vol
        d[f'ap{i}'] = round(ap1+(i-1)*0.001, 6); d[f'av{i}'] = vol
    return pd.Series(d, name=93000000)

def make_ob(): ob = OrderBook(); ob.init_from_snapshot(make_snapshot()); return ob

PASS, FAIL = '✓', '✗'
results = []

def check(name, condition):
    status = PASS if condition else FAIL
    results.append((status, name))
    print(f'  {status}  {name}')

print('✓ 测试工具定义完成')

✓ 测试工具定义完成


In [3]:
# ── Cell 3：Data Quality Assertions ──────────────────────────────────────────
print('═'*50)
print('Data Quality Assertions')
print('═'*50)

ob = make_ob()
snap = ob.to_snapshot()

check('bp1/ap1 初始化后不为 NaN',          not math.isnan(ob.bp1) and not math.isnan(ob.ap1))
check('所有买盘价格 > 0',                   all(snap[f'bp{i}'] > 0 for i in range(1,11)))
check('所有卖盘价格 > 0',                   all(snap[f'ap{i}'] > 0 for i in range(1,11)))
check('所有买盘量 > 0',                     all(snap[f'bv{i}'] > 0 for i in range(1,11)))
check('所有卖盘量 > 0',                     all(snap[f'av{i}'] > 0 for i in range(1,11)))
check('价差 > 0',                           ob.spread > 0)
check('买盘价格严格递减',                   all(snap[f'bp{i}'] > snap[f'bp{i+1}'] for i in range(1,10)))
check('卖盘价格严格递增',                   all(snap[f'ap{i}'] < snap[f'ap{i+1}'] for i in range(1,10)))
check('bp1 < ap1',                          ob.bp1 < ob.ap1)
check('mid_price 在 bp1 与 ap1 之间',       ob.bp1 < ob.mid_price < ob.ap1)
check('imbalance 在 [-1, 1] 区间',          -1.0 <= ob.imbalance <= 1.0)

ob2 = make_ob()
ob2.apply_order(Order(Index=93001000, side='buy', price=2.948, quantity=500, order_type='limit'))
check('apply_order 后 timestamp 更新',      ob2.timestamp == 93001000)

ob3 = make_ob()
ob3.apply_trade(Trade(Index=93001500, price=2.951, quantity=200))
check('apply_trade 后 timestamp 更新',      ob3.timestamp == 93001500)

══════════════════════════════════════════════════
Data Quality Assertions
══════════════════════════════════════════════════
  ✓  bp1/ap1 初始化后不为 NaN
  ✓  所有买盘价格 > 0
  ✓  所有卖盘价格 > 0
  ✓  所有买盘量 > 0
  ✓  所有卖盘量 > 0
  ✓  价差 > 0
  ✓  买盘价格严格递减
  ✓  卖盘价格严格递增
  ✓  bp1 < ap1
  ✓  mid_price 在 bp1 与 ap1 之间
  ✓  imbalance 在 [-1, 1] 区间
  ✓  apply_order 后 timestamp 更新
  ✓  apply_trade 后 timestamp 更新


In [4]:
# ── Cell 4：核心逻辑测试 ──────────────────────────────────────────────────────
print('═'*50)
print('核心逻辑测试')
print('═'*50)

ob = make_ob()
old_bp1 = ob.bp1
ob.apply_order(Order(Index=1, side='buy', price=2.940, quantity=500, order_type='limit'))
check('买单低于 bp1，bp1 不变',             ob.bp1 == old_bp1)

ob = make_ob()
ob.apply_order(Order(Index=1, side='buy', price=2.9505, quantity=300, order_type='limit'))
check('买单高于 bp1，bp1 更新',             abs(ob.bp1 - 2.9505) < 1e-6 and ob.bv1 == 300)

ob = make_ob(); old_bv1 = ob.bv1
ob.apply_order(Order(Index=1, side='buy', price=2.950, quantity=200, order_type='limit'))
check('买单等于 bp1，bv1 累加',             ob.bv1 == old_bv1 + 200)

ob = make_ob(); old_ap1 = ob.ap1
ob.apply_order(Order(Index=1, side='sell', price=2.960, quantity=500, order_type='limit'))
check('卖单高于 ap1，ap1 不变',             ob.ap1 == old_ap1)

ob = make_ob()
ob.apply_order(Order(Index=1, side='sell', price=2.9505, quantity=400, order_type='limit'))
check('卖单低于 ap1，ap1 更新',             abs(ob.ap1 - 2.9505) < 1e-6 and ob.av1 == 400)

ob = make_ob(); old_bp1 = ob.bp1
ob.apply_order(Order(Index=1, side='buy', price=2.960, quantity=500, order_type='market'))
check('market 单被跳过',                    ob.bp1 == old_bp1)

ob = make_ob(); old_ap1 = ob.ap1
ob.apply_order(Order(Index=1, side='sell', price=2.940, quantity=500, order_type='stop'))
check('stop 单被跳过',                      ob.ap1 == old_ap1)

ob = make_ob(); old_av1 = ob.av1
ob.apply_trade(Trade(Index=1, price=2.951, quantity=100))
check('主动买入，卖盘扣减量',               ob.av1 == old_av1 - 100)

ob = make_ob(); old_bv1 = ob.bv1
ob.apply_trade(Trade(Index=1, price=2.950, quantity=100))
check('主动卖出，买盘扣减量',               ob.bv1 == old_bv1 - 100)

ob = make_ob(); old_ap2 = ob.to_snapshot()['ap2']
ob.apply_trade(Trade(Index=1, price=2.951, quantity=ob.av1))
check('卖一被完全吃掉，ap1 移到 ap2',       abs(ob.ap1 - old_ap2) < 1e-6)

ob = make_ob()
ob.apply_trade(Trade(Index=1, price=2.951, quantity=100))
check('成交后 last_price 更新',             ob.last_price == 2.951)

ob = make_ob(); old_vol = ob.cum_volume
ob.apply_trade(Trade(Index=1, price=2.951, quantity=300))
check('成交后 cum_volume 累加',             ob.cum_volume == old_vol + 300)

ob = make_ob(); old_amt = ob.cum_amount
ob.apply_trade(Trade(Index=1, price=2.951, quantity=200))
check('成交后 cum_amount 累加',             abs(ob.cum_amount - (old_amt + 2.951*200)) < 1e-4)

══════════════════════════════════════════════════
核心逻辑测试
══════════════════════════════════════════════════
  ✓  买单低于 bp1，bp1 不变
  ✓  买单高于 bp1，bp1 更新
  ✓  买单等于 bp1，bv1 累加
  ✓  卖单高于 ap1，ap1 不变
  ✓  卖单低于 ap1，ap1 更新
  ✓  market 单被跳过
  ✓  stop 单被跳过
  ✓  主动买入，卖盘扣减量
  ✓  主动卖出，买盘扣减量
  ✓  卖一被完全吃掉，ap1 移到 ap2
  ✓  成交后 last_price 更新
  ✓  成交后 cum_volume 累加
  ✓  成交后 cum_amount 累加


In [5]:
# ── Cell 5：Edge Case 测试 ────────────────────────────────────────────────────
print('═'*50)
print('Edge Case 测试')
print('═'*50)

# 买盘为空
ob = OrderBook()
d = {'last': 2.951, 'volume': 0, 'amount': 0.0}
for i in range(1,11): d[f'bp{i}']=0.0; d[f'bv{i}']=0; d[f'ap{i}']=round(2.951+(i-1)*0.001,6); d[f'av{i}']=1000
ob.init_from_snapshot(pd.Series(d, name=93000000))
check('买盘为空，spread 为 NaN',            math.isnan(ob.spread))
check('买盘为空，mid_price 为 NaN',         math.isnan(ob.mid_price))

# 卖盘为空
ob = OrderBook()
d = {'last': 2.950, 'volume': 0, 'amount': 0.0}
for i in range(1,11): d[f'bp{i}']=round(2.950-(i-1)*0.001,6); d[f'bv{i}']=1000; d[f'ap{i}']=0.0; d[f'av{i}']=0
ob.init_from_snapshot(pd.Series(d, name=93000000))
check('卖盘为空，spread 为 NaN',            math.isnan(ob.spread))

# 双边为空时成交不 crash
ob = OrderBook()
d = {'last': 2.950, 'volume': 0, 'amount': 0.0}
for i in range(1,11): d[f'bp{i}']=0.0; d[f'bv{i}']=0; d[f'ap{i}']=0.0; d[f'av{i}']=0
ob.init_from_snapshot(pd.Series(d, name=93000000))
try:
    ob.apply_trade(Trade(Index=1, price=2.951, quantity=100))
    check('双边为空时成交不 crash',          True)
except Exception as e:
    check('双边为空时成交不 crash',          False)

# 超额成交不产生负数
ob = make_ob()
ob.apply_trade(Trade(Index=1, price=2.951, quantity=ob.av1 + 99999))
check('超额成交，av1 不为负数',              ob.av1 >= 0)

# 重复初始化清除旧状态
ob = OrderBook()
ob.init_from_snapshot(make_snapshot(bp1=3.000, ap1=3.001))
ob.init_from_snapshot(make_snapshot(bp1=2.950, ap1=2.951))
check('重复初始化，旧状态被清除',            abs(ob.bp1 - 2.950) < 1e-6)

# 浮点陷阱
check('浮点陷阱：2.950+0.001 == 2.951',     _to_int_price(2.950+0.001) == _to_int_price(2.951))

# 卖盘为空时新卖单成为卖一
ob = OrderBook()
d = {'last': 2.950, 'volume': 0, 'amount': 0.0}
for i in range(1,11): d[f'bp{i}']=round(2.950-(i-1)*0.001,6); d[f'bv{i}']=1000; d[f'ap{i}']=0.0; d[f'av{i}']=0
ob.init_from_snapshot(pd.Series(d, name=93000000))
ob.apply_order(Order(Index=1, side='sell', price=2.951, quantity=500, order_type='limit'))
check('卖盘为空时新卖单成为 ap1',            abs(ob.ap1 - 2.951) < 1e-6 and ob.av1 == 500)

# 档位不足补齐
ob = OrderBook()
d = {'last': 2.950, 'volume': 0, 'amount': 0.0}
for i in range(1,11):
    d[f'bp{i}'] = round(2.950-(i-1)*0.001,6) if i<=3 else 0.0; d[f'bv{i}'] = 1000 if i<=3 else 0
    d[f'ap{i}'] = round(2.951+(i-1)*0.001,6) if i<=3 else 0.0; d[f'av{i}'] = 1000 if i<=3 else 0
ob.init_from_snapshot(pd.Series(d, name=93000000))
snap = ob.to_snapshot()
check('档位不足时 0 补齐',                  all(snap[f'bp{i}']==0.0 and snap[f'bv{i}']==0 for i in range(4,11)))

══════════════════════════════════════════════════
Edge Case 测试
══════════════════════════════════════════════════
  ✓  买盘为空，spread 为 NaN
  ✓  买盘为空，mid_price 为 NaN
  ✓  卖盘为空，spread 为 NaN
  ✓  双边为空时成交不 crash
  ✓  超额成交，av1 不为负数
  ✓  重复初始化，旧状态被清除
  ✓  浮点陷阱：2.950+0.001 == 2.951
  ✓  卖盘为空时新卖单成为 ap1
  ✓  档位不足时 0 补齐


In [6]:
# ── Cell 6：Benchmark 测试 ────────────────────────────────────────────────────
print('═'*50)
print('Benchmark 测试')
print('═'*50)

import random
import time   # 🌟 修复报错的核心：导入 time 模块测速
random.seed(42)

def make_events(n):
    events = []
    for i in range(n):
        ts = 93000000 + i * 10
        if i % 3 == 0: events.append(Trade(Index=ts, price=2.951, quantity=100))
        else:
            side  = 'buy' if i % 2 == 0 else 'sell'
            price = 2.948 if side == 'buy' else 2.953
            events.append(Order(Index=ts, side=side, price=price, quantity=200, order_type='limit'))
    return events

for n, limit_ms in [(10_000, 500), (100_000, 5000)]:
    ob = make_ob()
    events = make_events(n)
    t0 = time.perf_counter()
    for e in events:
        if hasattr(e, 'order_type'): ob.apply_order(e)
        else: ob.apply_trade(e)
    ms = (time.perf_counter() - t0) * 1000
    ok = ms < limit_ms
    check(f'{n:,} 笔流水 < {limit_ms} ms（实际 {ms:.1f} ms）', ok)

ob = make_ob()
t0 = time.perf_counter()
for _ in range(1000): ob.to_snapshot()
ms = (time.perf_counter() - t0) * 1000
check(f'to_snapshot() x1000 < 100 ms（实际 {ms:.1f} ms）', ms < 100)

══════════════════════════════════════════════════
Benchmark 测试
══════════════════════════════════════════════════
  ✓  10,000 笔流水 < 500 ms（实际 58.5 ms）
  ✓  100,000 笔流水 < 5000 ms（实际 427.1 ms）
  ✓  to_snapshot() x1000 < 100 ms（实际 11.3 ms）


In [7]:
# ── Cell 7：汇总结果 ──────────────────────────────────────────────────────────
total  = len(results)
passed = sum(1 for s, _ in results if s == PASS)
failed = total - passed
print('\n' + '═'*50)
print(f'  测试结果：{passed}/{total} 通过')
if failed:
    print(f'  ✗ 失败 {failed} 项：')
    for s, n in results:
        if s == FAIL: print(f'      ✗ {n}')
else:
    print('  🎉 全部通过！')
print('═'*50)


══════════════════════════════════════════════════
  测试结果：38/38 通过
  🎉 全部通过！
══════════════════════════════════════════════════
